In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
!pip install -q roboflow ultralytics lapx


In [ ]:
import os

# configuration
BASE_DIR = "/content/drive/MyDrive/YOLO_trained_models"
PROJECT_NAME = "yolo_car_18_640_150epochs"

ROBOLFOW_API_KEY = "YOUR_API"

os.makedirs(BASE_DIR, exist_ok=True)

In [ ]:
from roboflow import Roboflow

# downloading roboflow dataset
rf = Roboflow(api_key = ROBOLFOW_API_KEY)
project = rf.workspace("archie-junio-dxv5t").project("car-detection-model-bwjpb")
version = project.version(18)
dataset = version.download("yolo26")


In [ ]:
from ultralytics import YOLO

# model training
model = YOLO("yolo26n.pt")

model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=150,
    imgsz=640,
    batch=-1,
    patience=20,
    project = BASE_DIR,
    name = PROJECT_NAME,
    save=True,
    save_period=15
)

In [ ]:
# display results, for more search in /PROJECT_NAME folder
from IPython.display import Image
Image(filename=f'{BASE_DIR}/{PROJECT_NAME}/results.png', width=800)
Image(filename=f'{BASE_DIR}/{PROJECT_NAME}/val_batch0_pred.jpg', width=1000)

In [ ]:
import cv2
from ultralytics import YOLO

# tracking from custom model
custom_model = YOLO(f"{BASE_DIR}/{PROJECT_NAME}/weights/best.pt")
video_path = "traffic.mp4"
output_path = "test_video_18.mp4"

cap = cv2.VideoCapture(video_path)

w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))
out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # tracking
    results = custom_model.track(frame, imgsz=2560, persist=True, conf=0.1, tracker="bytetrack.yaml")

    annotated_frame = results[0].plot()
    out.write(annotated_frame)

cap.release()
out.release()
print(f"Video saved: {output_path}")

In [ ]:
from PIL import Image

# one frame display
target_frame_index = 460

cap_check = cv2.VideoCapture(output_path)
cap_check.set(cv2.CAP_PROP_POS_FRAMES, target_frame_index)
ret, frame = cap_check.read()
cap_check.release()

if ret:
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(frame_rgb)
    display(img)
else:
    print(f"Error: Frame {target_frame_index} not found.")